# 🚀 yom Advanced Tutorial

**Debugging**, **Multi-Agent**, **Telegram Bot**, and **Advanced Patterns**!

## 1. Setup

In [ ]:
!pip install https://github.com/Niceiyke/yom-agent/releases/download/v0.1.1/yom-0.1.1-py3-none-any.whl --quiet

import os
os.environ['MINIMAX_API_KEY'] = 'sk-cp-q0jMznBbZY6vy7w8QnhhABqv_zUmMXAQutbp1TLGQRQvBVCVYTfdPVwA--TnJR_RBGAdwOpkLzrFJBvhvXfyLjTqFpeMEzw4JCVXza2zFBgWC_oqbZ2DnHw'

## 2. Debug Mode

Trace and debug agent behavior!

In [ ]:
from yom.debug import enable_debug, trace, get_recorder

# Enable debug output
enable_debug()

# Use trace context manager
with trace("my_operation") as ctx:
    from yom import Agent
    agent = Agent()
    result = agent.run_sync("What is 1+1?")

# View traces
recorder = get_recorder()
print(f"\nRecorded {len(recorder.events)} events")
for event in recorder.events[:5]:
    print(f"  - {event}")

## 3. Multiple Providers

Use different AI providers!

In [ ]:
from yom import Agent
from yom.providers import OllamaProvider, NVIDIAProvider

# MiniMax (default with MINIMAX_API_KEY)
agent_minimax = Agent()

# Ollama (local model)
# agent_ollama = Agent(provider=OllamaProvider(model="llama3"))

# NVIDIA NIM
# agent_nvidia = Agent(provider=NVIDIAProvider(
#     model="qwen/qwen3-coder-480b-a35b-instruct"
# ))

result = agent_minimax.run_sync("Reply with just 'Hello!'")
print(result)

## 4. Tool Versioning

Track and rollback tool versions!

In [ ]:
from yom.plugins import ToolVersionRegistry

registry = ToolVersionRegistry()

# Register v1
def tool_v1():
    return "v1.0"

registry.register("my_tool", tool_v1, "1.0.0")

# Register v2
def tool_v2():
    return "v2.0"

registry.register("my_tool", tool_v2, "2.0.0", rollback=tool_v1)

# Get latest
tv = registry.get("my_tool")
print(f"Version: {tv.version}, Result: {tv.func()}")

# Rollback if needed
registry.rollback("my_tool")
tv = registry.get("my_tool")
print(f"After rollback: {tv.version}")

## 5. State Inspection

Inspect agent state for debugging!

In [ ]:
from yom.debug import inspect_state
from yom import Agent

agent = Agent()

# Inspect runtime state
info = inspect_state(agent._runtime)
print(f"Runtime type: {info['type']}")
print(f"Attributes: {list(info.get('attrs', {}).keys())[:5]}")

## 6. Test Suite

Run comprehensive tests on your agent!

In [ ]:
import asyncio
from yom.testing import TestSuite, AgentTestCase, run_test_suite
from yom.testing import fake_agent

# Create test suite
suite = TestSuite(
    name="greeting_tests",
    cases=[
        AgentTestCase(
            name="basic_greeting",
            prompt="Say hello",
            expected_contains=["hello", "hi", "hey"]
        ),
        AgentTestCase(
            name="ask_name",
            prompt="What's your name?",
            expected_contains=["yom", "agent", "assistant"]
        ),
    ]
)

# Run tests with fake agent
agent = fake_agent("I am a helpful AI assistant.")
results = asyncio.run(run_test_suite(agent, suite))

print(f"Passed: {results['passed']}")
print(f"Failed: {results['failed']}")
for test in results['tests']:
    status = "✅" if test['passed'] else "❌"
    print(f"  {status} {test['name']}")

## 7. Plugin System

Extend functionality with plugins!

In [ ]:
from yom.plugins import YomApp, ToolPlugin
from yom import tool

# Create a custom plugin
class MyPlugin(ToolPlugin):
    name = "my-plugin"
    version = "1.0.0"
    
    @staticmethod
    @tool(name="greet")
    def greet(name: str) -> str:
        return f"Hello, {name}!"
    
    def get_tools(self):
        return [self.greet]

# Use plugin
app = YomApp()
app.plugin_manager.register_plugin(MyPlugin())

from yom import Agent
agent = Agent(tools=app.get_tools())
result = agent.run_sync("Use the greet tool to greet Alice")
print(result)

## 8. Telegram Bot Setup

**Note:** This requires a Telegram bot token and a public server.

### Quick Setup:

In [ ]:
# Setup Telegram bot
# 1. Message @BotFather on Telegram
# 2. Get your bot token
# 3. Replace TOKEN below

from yom.toolsets.telegram import TelegramBot
from yom import Agent

TOKEN = "YOUR_TELEGRAM_BOT_TOKEN"

agent = Agent(tools=["core"])
bot = TelegramBot(token=TOKEN, agent=agent)

# Start polling (blocks)
# await bot.poll()

## 🎯 Advanced Challenge

Build a multi-tool agent that:
1. Uses HTTP to fetch data
2. Processes with custom tool
3. Stores result in session
4. Can answer questions about stored data